In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Colabのシークレットからトークンを読み込んで自動ログイン
login(token=userdata.get('HF_TOKEN'))

In [ ]:
olmo = AutoModelForCausalLM.from_pretrained("allenai/Olmo-3-1025-7B")
tokenizer = AutoTokenizer.from_pretrained("allenai/Olmo-3-1025-7B")
message = ["Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q., Answer: "]
inputs = tokenizer(message, return_tensors='pt', return_token_type_ids=False)
# optional verifying cuda
inputs = {k: v.to('cuda') for k,v in inputs.items()}
olmo = olmo.to('cuda')
response = olmo.generate(**inputs, max_new_tokens=100, do_sample=True, top_k=0, temperature=1.0, top_p=0.7)
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model_id = "allenai/OLMo-3-1025-7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
subject = "all"
batch_size = 2

cuda


In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate

In [ ]:
print(f"loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quantization_config
)
model.eval()

loading allenai/OLMo-3-1025-7B...


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Olmo3ForCausalLM(
  (model): Olmo3Model(
    (embed_tokens): Embedding(100278, 4096, padding_idx=100277)
    (layers): ModuleList(
      (0-31): 32 x Olmo3DecoderLayer(
        (self_attn): Olmo3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Olmo3RMSNorm((4096,), eps=1e-06)
          (k_norm): Olmo3RMSNorm((4096,), eps=1e-06)
        )
        (mlp): Olmo3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): 

In [ ]:
print(f"loading MMLU ({subject})...")
dataset = load_dataset("cais/mmlu", subject, split="test")

loading MMLU (all)...


In [ ]:
choices = ["A", "B", "C", "D"]
choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
prompt_template = "Question: {question}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer: "

In [ ]:
prompts = []
answers = []
for sample in dataset:
    p = prompt_template.format(
        question=sample['question'],
        a=sample['choices'][0],
        b=sample['choices'][1],
        c=sample['choices'][2],
        d=sample['choices'][3]
    )
    prompts.append(p)
    answers.append(sample['answer'])

correct_count = 0
total_count = len(dataset)

print("評価を開始します...")
for i in tqdm(range(0, total_count, batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    batch_answers = answers[i:i+batch_size]

    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(**inputs)

    # バッチ内の各問題に対する正誤判定
    for j in range(len(batch_prompts)):
        # padding_side="left" なので、常に一番右(-1)が最後のトークンになる
        next_token_logits = outputs.logits[j, -1, :]
        choice_logits = [next_token_logits[cid].item() for cid in choice_ids]
        predicted_index = choice_logits.index(max(choice_logits))

        if predicted_index == batch_answers[j]:
            correct_count += 1

評価を開始します...


  0%|          | 0/7021 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
accuracy = (correct_count / total_count) * 100
print("\n" + "="*40)
print(f"モデル: {model_id}")
print(f"科目: {subject}")
print(f"正解数: {correct_count} / {total_count}")
print(f"正答率 (Accuracy): {accuracy:.2f}%")
print("="*40)

In [ ]:
dataset = load_dataset("cais/mmlu", subject, split="test")
choices = ["A", "B", "C", "D"]
choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
prompt_template = "Question: {question}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer: "

prompts = []
answers = []
for sample in dataset:
    p = prompt_template.format(
        question=sample['question'],
        a=sample['choices'][0],
        b=sample['choices'][1],
        c=sample['choices'][2],
        d=sample['choices'][3]
    )
    prompts.append(p)
    answers.append(sample['answer'])

correct_count = 0
total_count = len(dataset)
skipped_count = 0  # スキップした問題の数

# ==========================================
# 4. バッチ評価ループ（絶対に止まらない仕様）
# ==========================================
print("評価を開始します...")
for i in tqdm(range(0, total_count, batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    batch_answers = answers[i:i+batch_size]

    # max_lengthを設定し、異常な長さの文章をカットしてメモリ爆発を防ぐ
    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048  # ここが超重要
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    try:
        with torch.inference_mode():
            outputs = model(**inputs)

        # 正誤判定
        for j in range(len(batch_prompts)):
            next_token_logits = outputs.logits[j, -1, :]
            choice_logits = [next_token_logits[cid].item() for cid in choice_ids]
            predicted_index = choice_logits.index(max(choice_logits))

            if predicted_index == batch_answers[j]:
                correct_count += 1

    except torch.cuda.OutOfMemoryError:
        # メモリ不足が起きてもクラッシュさせず、スキップして進む
        skipped_count += len(batch_prompts)
        pass

    finally:
        # バッチ終了ごとにメモリを強制的に解放する（ゴミ拾い）
        if 'inputs' in locals(): del inputs
        if 'outputs' in locals(): del outputs
        torch.cuda.empty_cache()
        gc.collect()

# ==========================================
# 5. 結果発表
# ==========================================
# スキップされた分は母数から引いて正確な正答率を出す
valid_total = total_count - skipped_count
accuracy = (correct_count / valid_total) * 100 if valid_total > 0 else 0

print("\n" + "="*40)
print(f"モデル: {model_id}")
print(f"評価完了数: {valid_total} / {total_count} (スキップ: {skipped_count})")
print(f"正解数: {correct_count} / {valid_total}")
print(f"正答率 (Accuracy): {accuracy:.2f}%")
print("="*40)

評価を開始します...


 42%|████▏     | 1471/3511 [1:37:56<7:18:03, 12.88s/it]